## 1) 앙상블 학습
* 여러 개의 분류기(classifier)을 생성하고 그 예측을 결합해 보다 정확한 최종 예측을 도출하는 기법 -> 신뢰성이 높은 예측값 얻고자
* 대표적 알고리즘 : 랜덤 포레스트, 그래디언트 부스팅 알고리즘, XGBoost, LightGBM, Stacking 등
* 정형 데이터의 분류가 회귀에서 예측 성능 ㅜ띠어남
* 유형 : 보팅, 배깅, 부스팅
  * 보팅(voting) : 서로 다른 알고리즘을 가진 분류기를 결합하는 것
  * 배깅(bagging)
    * 각각의 분류기가 모두 같은 유형의 알고리즘 기반. 데이터 샘플링을 서로 다르게 가져가며 학습을 수행해 보팅을 수행하는 것
    * 랜덤 포레스트 알고리즘이 대표적
    * 부트 스트래핑(bootstrapping) 분할 방식 : 개별 분류기에 할당된 학습 데이터는 원본 학습 데이터를 샘플링해 추출하는데, 이렇게 개별 classifier에게 데이터를 샘플링해서 추출한느 방식
    * 개별 분류기가 부트 스트래핑 방식으로 샘플링된 데이터 세트에 대해 학습을 통해 개별적 예측을 수행한 결과를 보팅을 통해 최종 예측 결과를 선정함
    * 교차검증과는 다르게, 데이터 세트 간 중첩을 허용함
  * 부스팅(boosting)
    * 여러 개의 분류기가 순차적으로 학습을 수행하되, 앞에서 학습한 분류기가 예측이 틀린 데이터에 대해서는 올바르게 예측하도록 다음 분류기에게는 가중치(weight)를 부여하며 학습과 예측 진행
    * 예 : 그래디언트 부스트, XGBoost, LightGBM 등
  * 스태킹
    * 여러 가지 다른 모델의 예측 결괏값을 다시 학습 데이터로 만들어 다른 모델(메타 모델)로 재학습시켜 결과를 예측하는 방법

## 2) 보팅 유형 : hard voting, soft voting
* 하드 보팅을 이용한 분류기(classifier)
  * 예측 결괏값 중 다수의 분류기가 결정한 예측값을 최종 보팅 결괏값으로 선정하는 것
  * 다수결의 원칙과 비슷함
* 소프트 보팅을 이용한 분류기
  * 분류기들의 레이블 값 결정 확률을 모두 더하고 이를 평균해, 이들 중 확률이 가장 높은 레이블 값을 최종 보팅 결괏값으로 선정
  * 일반적으로 소프트 보팅이 보팅 방법으로 적용됨(예측 성능 좋음)

## 3) 보팅 분류기(Voting Classifier)
* 사이킷런에서는 보팅 방식의 앙상블을 구현한 VotingClassifier 클래스 제공

## 4) 보팅 방식의 앙상블 실습 : 위스콘신 유방암 데이터 세트 예측 분석
* 유방암의 악성종양, 양성종양 여부를 결정하는 이진 분류 데이터 세트
* 종양의 크기, 모양 등 형태와 관련한 많은 피처 보유
* load_breast_cancer() 함수

In [3]:
### 로지스틱 회귀와 KNN을 기반으로 보팅 분류기 만들기
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 불러오기
cancer = load_breast_cancer()
data_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
data_df.head(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [6]:
# 개별 모델은 로지스틱 회귀와 KNN
lr_clf = LogisticRegression()
knn_clf = KNeighborsClassifier(n_neighbors=8)

# 개별 모델을 소프트 보팅 기반의 앙상블 모델로 구현한 분류기
vo_clf = VotingClassifier(estimators=[('LR', lr_clf), ('KNN', knn_clf)], voting='soft')
x_train, x_test, y_train, y_test = train_test_split(cancer.data, cancer.target, test_size=0.2, random_state=156)

# VotingClassifier 학습/예측/평가
vo_clf.fit(x_train, y_train)
pred = vo_clf.predict(x_test)
print('Voting 분류기 정확도 : {0:.4f}'.format(accuracy_score(y_test, pred)))

# 개별 모델의 학습/예측/평가 (<->Voting은 여러 개의 분류기가 결합한 형태)
classifiers = [lr_clf, knn_clf]
for classifier in classifiers:
    classifier.fit(x_train, y_train)
    pred = classifier.predict(x_test)
    class_name = classifier.__class__.__name__
    print('{0} 정확도 : {1:.4f}'.format(class_name, accuracy_score(y_test, pred)))

Voting 분류기 정확도 : 0.9474
LogisticRegression 정확도 : 0.9386
KNeighborsClassifier 정확도 : 0.9386


C:\Users\lumos\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\lumos\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

## 5) ML 모델의 유연성
* 편향-분산 트레이드오프는 ML 모델이 극복해야 할 중요 과제임
* 보팅과 스태킹 등은 서로 다른 알고리즘을 기반으로 하고 있음. 배깅과 부스팅은 대부분 결정 트리 알고리즘을 기반으로 함
* 앙상블 학습에서는 결정 트리 알고리즘의 단점을 수십~수천 개의 매우 많은 분류기를 결합해 다양항 상황 학습하게 하여 극복 & 편향-분산 트레이드 오프 효과 극대화